# DiaFuse-PRC++ — one-click experimental run

**For the person running this notebook:** select *Runtime → Change runtime type → A100 GPU*, then *Runtime → Run all*.
Allow Google Drive access when asked. Then wait until the last line shows **RUN COMPLETED** (this takes more than one day).

If Colab disconnects or stops before that, open this notebook again and choose *Runtime → Run all*: the run resumes where it stopped and nothing already finished is recomputed.

What this notebook runs, in the order of the pre-registered protocol (`configs/protocol.json`):
1. data, split and model checks;
2. **B12 — DiaFuse-PRC (Setting A)** reproduction with the original trainer (`original/code/diafuse_run.py`);
3. **B11 — Robust + calibrated (Shao et al. style)** reproduction with the original `original/code/finetune.py`;
4. screening of DiaFuse-PRC++ **Setting A** (`src/diafuse_pp_setting_A.py`, `configs/B12_setting_A/`) and **Setting B, LoRA** (`src/diafuse_pp_setting_B_lora.py`, `configs/B12_setting_B_lora/`) on the development sets only;
5. selection, final runs (10 seeds), ablations (5 seeds), evaluation, statistics, tables, figures, report and archive.

Each model is written to its own folder in `My Drive/DiaFuse_PRCPP_RUN/results/`: `B11/`, `B12_setting_A/`, `B12pp_setting_A/`, `B12_setting_B_lora/`.

Do not change anything in the cells below.

In [ ]:
# ======================= CONTROL (do not modify) =======================
REPO_URL         = "https://github.com/SAMAWELJABALLI/diafuse-prc-pp.git"   # empty string: the repository folder is searched in My Drive instead
REPO_BRANCH      = "main"
RUN_MODE         = "FULL"      # "FULL" = complete protocol, "PRIORITY" = reduced protocol (decided by the project owner only)
RESUME           = True        # continue an interrupted run in the same Drive folder
GPU_REQUIRED     = True
RUN_REPRODUCTION = True
RUN_SCREENING    = True
RUN_FINAL        = True
RUN_ABLATIONS    = True
GENERATE_REPORT  = True
ZIP_RESULTS      = True
RUN_DIR_NAME     = "DiaFuse_PRCPP_RUN"   # created in MyDrive; all outputs are written there

In [ ]:
import os, sys, glob, json, shutil, subprocess, importlib.metadata as md, datetime

def banner(msg):
    print("=" * 60 + "\n" + msg + "\n" + "=" * 60, flush=True)

def fatal(msg):
    banner("FATAL ERROR\n" + msg)
    raise SystemExit(msg)

def sh(*cmd, cwd=None):
    r = subprocess.run(list(cmd), cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        fatal(f"command failed: {' '.join(cmd)}\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()

def mount_drive():
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        fatal("Google Drive is not available.")
    return "/content/drive/MyDrive"

def is_repo_root(d):
    return all(os.path.exists(os.path.join(d, x)) for x in ("scripts/run_one_click.py", "original/code/MD5SUMS_unmodified.txt", "configs/protocol.json"))

def get_code(mydrive, run_dir):
    """Repository checkout used for the whole run. With REPO_URL, the commit used by the first session is recorded on
    Drive and every later session checks out exactly that commit (no code change in the middle of a run)."""
    pin = os.path.join(run_dir, "state", "code_commit.txt")
    if REPO_URL:
        dst = "/content/DiaFuse-PRC-PP"
        if not is_repo_root(dst):
            shutil.rmtree(dst, ignore_errors=True)
            sh("git", "clone", "--quiet", "--branch", REPO_BRANCH, REPO_URL, dst)
        if os.path.exists(pin):
            commit = open(pin).read().strip()
            sh("git", "fetch", "--quiet", "origin", cwd=dst); sh("git", "checkout", "--quiet", commit, cwd=dst)
        else:
            commit = sh("git", "rev-parse", "HEAD", cwd=dst)
            os.makedirs(os.path.dirname(pin), exist_ok=True); open(pin, "w").write(commit + "\n")
        print("REPOSITORY:", REPO_URL, "commit", commit)
        return dst
    hits = []
    for depth in ("", "*/", "*/*/", "*/*/*/"):
        for h in glob.glob(f"{mydrive}/{depth}*/scripts/run_one_click.py"):
            root = os.path.dirname(os.path.dirname(h))
            if is_repo_root(root) and root not in hits:
                hits.append(root)
    if not hits:
        fatal("The repository folder (containing 'scripts', 'src', 'configs' and 'original') was not found in My Drive.\n"
              "Copy the whole repository folder into My Drive and run again.")
    hits.sort(key=lambda r: os.path.getmtime(os.path.join(r, "scripts", "run_one_click.py")), reverse=True)
    if len(hits) > 1:
        print("WARNING: several copies of the repository were found; using the most recent one:", hits[0])
    return hits[0]

def gpu_check():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    except Exception:
        out = ""
    print("GPU:", out or "none")
    if GPU_REQUIRED and not out:
        fatal("FATAL: GPU required. Select Runtime > Change runtime type > A100 GPU, then Runtime > Run all.")
    return out

def install_requirements(req):
    for line in open(req):
        line = line.split("#", 1)
        spec, tag = line[0].strip(), (line[1].strip() if len(line) > 1 else "")
        if not spec:
            continue
        name = spec.split("==")[0]
        try:
            have = md.version(name)
        except md.PackageNotFoundError:
            have = None
        want = spec.split("==")[1] if "==" in spec else None
        if tag.startswith("strict") and have != want or (tag.startswith("missing") and have is None) or (tag.startswith("optional") and have is None):
            r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], capture_output=True, text=True)
            status = "installed" if r.returncode == 0 else ("FAILED (optional, ignored)" if tag.startswith("optional") else "FAILED")
            print(f"  {spec:28s} {status}")
            if status == "FAILED" and tag.startswith("strict"):
                print(f"  WARNING: {spec} could not be installed; the installed version {have} is used and recorded.")
        else:
            print(f"  {spec:28s} present ({have})")

def main():
    banner("DiaFuse-PRC++ one-click run")
    gpu_check()
    mydrive = mount_drive()
    run_dir = os.path.join(mydrive, RUN_DIR_NAME)
    if not RESUME and os.path.exists(run_dir):
        run_dir = run_dir + "_" + datetime.datetime.now().strftime("%Y%m%d-%H%M")
    project = get_code(mydrive, run_dir)
    work = "/content/pp_work"; local_pkg = "/content/pp_pkg"
    shutil.copytree(project, local_pkg, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))  # full copy: git status stays clean for the code-version record
    print("PROJECT ROOT:", project); print("RUN DIR (outputs on Google Drive):", run_dir)
    print("Installing missing dependencies (pinned list: configs/requirements_colab.txt)")
    install_requirements(os.path.join(local_pkg, "configs", "requirements_colab.txt"))
    try:
        r = subprocess.run([sys.executable, "-c", "import torch;print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
                           capture_output=True, text=True)
        print("torch / CUDA / available:", r.stdout.strip())
        if GPU_REQUIRED and "True" not in r.stdout:
            fatal("FATAL: GPU required (PyTorch does not see a CUDA device).")
    except Exception as ex:
        fatal(f"PyTorch is not usable: {ex}")
    cmd = [sys.executable, "-u", os.path.join(local_pkg, "scripts", "run_one_click.py"),
           "--run_dir", run_dir, "--work_dir", work, "--mode", RUN_MODE, "--resume", str(int(RESUME)),
           "--gpu_required", str(int(GPU_REQUIRED)), "--run_reproduction", str(int(RUN_REPRODUCTION)),
           "--run_screening", str(int(RUN_SCREENING)), "--run_final", str(int(RUN_FINAL)),
           "--run_ablations", str(int(RUN_ABLATIONS)), "--generate_report", str(int(GENERATE_REPORT)),
           "--zip_results", str(int(ZIP_RESULTS))] + os.environ.get("PP_ORCHESTRATOR_EXTRA_ARGS", "").split()  # empty on Colab (test hook)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=work if os.path.isdir(work) else "/content")
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    if code == 3:
        fatal("The run stopped on a FATAL error (see above). Send the folder logs/ and run_state.json of\n" + run_dir + "\nto the project owner.")
    if code == 4:
        banner("RUN INTERRUPTED\nOpen this notebook again and choose Runtime > Run all to resume.")
    return code

os.makedirs("/content/pp_work", exist_ok=True)
exit_code = main()